In [42]:
import pandas as pd
import yfinance as yf
import numpy as np
from scipy.stats import norm
import os
print(os.getcwd())

/Users/lydialichen/Desktop/quant-risk-engine/notebooks


In [43]:
def download_returns(tickers: list, period: str) -> pd.DataFrame: 
    """ 
    Input an array of stock labels and the lookback window and obtain a pandas 
    dataframe containing: rows = with daily periodicity, the number of observations up to the lookback window. 
    columns = the ticker 
    
    Parameters 
    ----------
    tickers: list 
        list containing stock labels 
    period:  str 
        defines the lookback period 
    
    Returns 
    ----------
    pd.DataFrame
        table containing the stock return at each observation point for the duration of the lookback period for all tickers contained in the tickers
        array
    """

    prices = yf.download(tickers= tickers, period = period)["Close"]
    returns = np.log((prices / prices.shift(1))) # the resulting dataframe has the same number of rows as the price dataframe 
    return returns.dropna() # final dataframe has the number of rows of the prices dataframe - 1 from the NaN value 


In [44]:
def calculate_portfolio_pnl(returns: pd.DataFrame, positions: dict) -> pd.Series: 
    """
    Input the returns panda DataFrame containing the daily realised returns, and long / short dollar position associated to each ticker, to output a 
    series of daily P&L valulues, which should then be sorted to find the VaR threshold, which is usuallly the value at the 99% 
    
    Parameters: 
    ----------
    returns: pandas dataframe 
        contains daily realised returns 
    positions: dictionary 
        contains each ticker mapped to its dollar position : if negative -> short position, if positive -> long position
    
    Returns: 
    ----------
    pd.Series
        a series containing the net P&L daily returns within the lookback period given all the different ticker positions

    """
    
    return returns.dot(pd.Series(positions))

In [45]:
def historical_var(pnl: pd.Series, confidence: float) -> float: 
    """
    Computes Historical VaR from a series of daily P & L values by sorting and selecting the loss at the specified 
    confidence level percentile. 

    Parameters: 
    -----------
    pnl: pd.Series
        Daily portfolio P & L values from calculate_portfolio_pnl() 
    
    confidence: float
        confidence level e.g 0.95 or 0.99 - probability that losses will 
        not exceed the VaR threshold on any given day 

    Returns: 
    -----------
    float: 
        VaR in dollars expressed as a positive loss 
    """

    sorted_pnl = pnl.sort_values() # sort pnl values in the P&L series from worst to best 
    index = int((1 - confidence) * len(sorted_pnl))
    sorted_pnl = pnl.sort_values()
    return -sorted_pnl.iloc[index]

In [46]:
def parametric_var(portfolio_value: float, daily_std: float, confidence:float, horizon: int = 1) -> float:
    """
    Inputs the portfolio value, the daily standard deviation, the confidence level and as default the horizon period is 1 day: returns the 1 day VaR by default 

    Parameters: 
    ----------
    portfolio_value: float 
        gross value of the portfolio given the (daily) P&L 
    daily_std: float    
        daily standard deviation of portfolio returns s 
    confidence: float   
        level of confidence (the width of the guarantee that the VaR is giving an accurate estimate of the maximum loss)
    horizon: int = 1
        time period duration of the computed parametric VaR 


    Returns: 
    ----------
    float:
        parametric VaR with a default 1 day period 
    """ 

    z = norm.ppf(confidence)
    VaR = z * daily_std * np.sqrt(horizon) * portfolio_value # Parametric VaR = z × σ × Portfolio Value × √T
    return VaR

In [47]:
def stress_test(positions: dict, scenarios: dict) -> dict: 
    """
    Multiplies each position with its associated percentage move across each scenario and returns a dictionary of single float of the dollar P&L under that scenario 

    Parameters: 
    -----------
    positions: dict 
        each ticker mapped to its dollar position
    scenario: dict
        each different scenario to stress test mapped to the percentage move of each ticker 
    

    Returns: 
    -----------
    dict: 
        mapping of scenario to net P & L from that scenario 

    """
    return {
        scenario_name: pd.Series(moves).dot(pd.Series(positions))
        for scenario_name, moves in scenarios.items()
    }
    

In [ ]:
def export_to_excel(
        returns:pd.DataFrame, 
        positions:dict, 
        pnl: pd.Series, 
        historical_var_99: float, 
        historical_var_95: float, 
        parametric_var_99: float, 
        parametric_var_95: float, 
        stress_results: dict, 
        filename: str = "/Users/lydialichen/Desktop/quant-risk-engine/outputs/var_report.xlsx"
) -> None: 
        """
        Parameters: 
        -----------
        returns: pd.DataFrame   
            data frame containing the stock returns realised at a daily rate within the user-defined lookback period
        positions: dict 
            contains the mapping of each ticker to its dollar position (long / short)
        pnl: pd.Series  
            series containing the profit and loss of each ticker over the entire lookback period 
        historical_var_99: float 
            historical VaR with confidence level 99% with a default horizon of 1 day 
        historical_var_95: float 
            historical VaR with confidence leel 95% ith a default horizon of 1 day 
        parametric_var_99: float 
            parametric VaR calculated using the z-score formula where z-score is a fixed value defined by the 99% confidence level 
        parametric_var_95: float 
            parametric VaR calculated using the z-score formula where z-score is a fixed value defined by the 95% confidence level 
        stress_results: dict
            dictionary containing the net P&L per scenario across the entire portfolio 
        filename: str
            filename of the excel file containing the outputs of the VaR report  

        Returns: 
        ----------
        Excel file with 4 sheets: 
            1. Portfolio Summary: positions table, historical VaR (95 and 99%), parametric VaR (95 and 99%), 10-day VaR (scaled)
            2. Historical P & L Distribution: daily P & L series sorted from worst to best, VaR threshold highlighted, Summary statistics (mean, std, min, max)
            3. Stress test Results: P & L under each scenario, breakdown by ticker, and comparison to VaR 
            4. Returns Data: raw daily returns for each ticker
          
        """
        os.makedirs(os.path.dirname(os.path.abspath(filename)), exist_ok=True)

        with pd.ExcelWriter(filename, engine = "openpyxl") as writer: 
            # with block ensures the file is saved and closed properly even if something goes wrong 
            # engine = "openpyxl" is specified because it is a Python library that reads and writes Excel .xlsx files. Without specifying .xlsx format, pandas might use a different engine that doesn't support all features
            
            # Sheet 1: Portfolio summary 

            ## Portfolio positions table 
            portfolio_df = pd.DataFrame({
                "Ticker" : list(positions.keys()),
                "Position ($)": list(positions.values()), 
                "Direction": ['Long' if v > 0 else "Short" for v in positions.values()]
            })

            ## VaR summary table 
            var_summary = pd.DataFrame({
                  "Metric" : ["Historical VaR 99%", "Historical VaR 95%", "Parametric VaR 99%", "Parametric VaR 95%", "10 day VaR 99% (scaled)"], 
                  "Value ($)": [historical_var_99, historical_var_95, parametric_var_99, parametric_var_95, historical_var_99 * np.sqrt(10)]
                  })
            
            portfolio_df.to_excel(writer, sheet_name = "Portfolio Summary", index = False, startrow = 0)
            var_summary.to_excel(writer, sheet_name = "Portfolio Summary", index = False, startrow = len(portfolio_df) + 3)

            # Sheet 2: Historical P & L Distribution

            ## daily P&L sorted worst to best across entire portfolio for days up to the lookback period s
            pnl_sorted = pnl.sort_values().reset_index()
            pnl_sorted.columns = ["Date", "Daily P&L ($)"]

            ## summary statistics of the profit and loss 
            summary_stats = pd.DataFrame({
                  "Statistic": ["Mean", "Std Dev", "Min", "Max", "VaR 99%"], 
                  "Value": [pnl.mean(), pnl.std(), pnl.min(), pnl.max(), historical_var_99]
            })

            pnl_sorted.to_excel(writer, sheet_name = "Historical PnL", index = False, startrow = 0)
            summary_stats.to_excel(writer, sheet_name= "Historical PnL", index = False, startrow = 0, startcol = 4 )

            # Sheet 3: Stress Test Results 
            stress_df = pd.DataFrame({
                  "Scenario": list(stress_results.keys()), 
                  "Portfolio P&L": list(stress_results.values()),
                  "vs Historical VaR 99%": [v / historical_var_99 for v in stress_results.values()]
            })

            stress_df.to_excel(writer, sheet_name = "Stress Tests", index = False)

            # Sheet 4: Returns Data 
            returns.to_excel(writer, sheet_name = "Returns Data", index = True)

In [ ]:
def main(): 
    # 1. Define tickers and positions 
    tickers = ["SPY","TLT","GLD","AAPL","MSFT" ]
    positions = {
    'SPY':  1_000_000,   # long $1m — S&P 500 ETF
    'TLT':    500_000,   # long $500k — 20-year Treasury ETF
    'GLD':    300_000,   # long $300k — investment grade corporate bond ETF
    'AAPL':    200_000,   # long $200k — gold ETF
    'MSFT':   -400_000    # short $400k — tech ETF
}

    # 2. Download returns 
    returns = download_returns(tickers, period = "1y")

    # 3. Calculate portfolio P&L 
    pnl = calculate_portfolio_pnl(returns, positions)

    # 4. Calculate daily std for parametric VaR 
    daily_std = pnl.std()

    # 5. Portfolio Value
    portfolio_value = sum(v for v in positions.values() if v > 0)
    
    # 6. Calculate Historical and Parametric VaR
    historical_var_99 = historical_var(pnl, 0.99) 
    historical_var_95 = historical_var(pnl, 0.95)

    parametric_var_99 = parametric_var(portfolio_value, daily_std,0.99, horizon=1) 
    parametric_var_95 = parametric_var(portfolio_value, daily_std,0.95, horizon=1) 
    
    # 7. Define stress scenarios 
    scenarios = {
        "2008": {
            "SPY": -0.4, 
            "TLT": 0.25, 
            "GLD": -0.15, 
            "AAPL": 0.05, 
            "MSFT": -0.45
        },
        "COVID": {
            "SPY": -0.35, 
            "TLT": 0.20, 
            "GLD": -0.10, 
            "AAPL": 0.03, 
            "MSFT": -0.30
        },
        "2022": {
            "SPY": -0.2, 
            "TLT": -0.3, 
            "GLD": -0.25, 
            "AAPL": -0.05, 
            "MSFT": -0.35
        },
    }

    # 8. Run stress tests 
    stress_results = stress_test(positions, scenarios)

    # 9. Export to Excel
    export_to_excel(returns, positions, pnl, historical_var_99, historical_var_95, parametric_var_99, parametric_var_95, stress_results, filename = "../outputs/var_report.xlsx")

if __name__ == "__main__": 
    main()